# Papapanagiotou Panagiotis

#### My basic setup code


In [3]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / "../module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {MODEL}")

OpenAI client ready - using model gpt-4o-mini


#### Θα χρησιμοποιήσω την συνάρτηση που φτιάξαμε.

In [5]:
def ask(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model:  str="gpt-4o-mini") -> str:
    """
    Send a prompt to the OpenAI Chat Completions API and return the assistant's reply.

    This helper function builds a chat message list, optionally including a system
    instruction, sends it to the specified model, and returns the generated text
    content from the first response choice.

    Args:
        prompt (str): The user prompt to send to the model.
        system_content (str, optional): Optional system-level instruction that
            defines the assistant's behavior, tone, or constraints. If provided,
            it is added as the first message in the conversation. Defaults to None.
        temperature (float, optional): Sampling temperature used to control
            randomness in the model's response. Lower values produce more
            deterministic outputs; higher values produce more creative or varied
            outputs. Defaults to 0.7.
        max_resp_tokens (int, optional): Maximum number of tokens allowed in the
            generated response. Defaults to 800.
        ai_model (str, optional): The model name to use for the request.
            Defaults to "gpt-4o-mini".

    Returns:
        str: The text content of the assistant's first response message.

    Raises:
        Exception: Propagates any exception raised by the OpenAI client request
            (for example, authentication errors, rate limits, invalid parameters,
            or network-related failures).

    Example of use:
         reply = ask("Explain recursion in simple terms.")
         print(reply)

        reply = ask(
             prompt="Write a haiku about Python.",
             system_content="You are a poetic assistant.",
             temperature=0.9,
             max_resp_tokens=100,
             ai_model="gpt-4o-mini"
        )        
        print(reply)
    """
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

display(Markdown(f"**The function has run**"))

**The function has run**

## Για να πειραματιστώ θα δώσω δύο λύσεις. Μία απλά με prompting και μία με response formating.

### Λύση 1η. `Μόνο Prompting`.

In [6]:
import json

prompt = '''Extract information from this text and return as a valid JSON:

Text: "Ο Γιάννης αγόρασε ένα κινητό και είναι πολύ ευχαριστημένος με την κάμερα αλλά όχι με την μπαταρία."

Return JSON fields: sentiment, positive_aspects, negative_aspects.
No additional properties.

Do NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON object, nothing else.

JSON:
'''

result = ask(prompt, temperature=0.0)

try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}\nNot valid JSON")





```json
{
  "sentiment": "mixed",
  "positive_aspects": [
    "\u03ba\u03ac\u03bc\u03b5\u03c1\u03b1"
  ],
  "negative_aspects": [
    "\u03bc\u03c0\u03b1\u03c4\u03b1\u03c1\u03af\u03b1"
  ]
}
```

### Το αφήνω επείτηδες το παραπάνω έτσι επειδή με έκανε εντύπωση τι συνέβει λόγο των Ελληνικών.

### Λύση 1η. Μόνο prompting (εξελληνισμένη 😅)

In [7]:
import json

prompt = '''Extract information from this text and return as a valid JSON:

Text: "Ο Γιάννης αγόρασε ένα κινητό και είναι πολύ ευχαριστημένος με την κάμερα αλλά όχι με την μπαταρία."

Return JSON fields: sentiment, positive_aspects, negative_aspects.
No additional properties.

Do NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON object, nothing else.

JSON:
'''

result = ask(prompt, temperature=0.0)

try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2, ensure_ascii=False)}\n```")) # Auto krataei tous Ellinikous xaraktires
except json.JSONDecodeError:
    print(f"Raw output: {result}\nNot valid JSON")

```json
{
  "sentiment": "mixed",
  "positive_aspects": [
    "κάμερα"
  ],
  "negative_aspects": [
    "μπαταρία"
  ]
}
```

### Λύση 2η. `Response formating`

In [8]:
response_json = client.chat.completions.create(
    model = MODEL,
    messages=[{'role':"user", 'content': prompt}],
    temperature=0.0,
    response_format={"type": "json_schema",
                        'json_schema': {
                            'name': 'sentiment_response',
                            'schema': {
                                'type': 'object',
                                'properties': {
                                    'sentiment': {'type': 'string'},
                                    'positive_aspects': {'type': 'string'},
                                    'negative_aspects': {'type': 'string'}
                                },
                                'required': ['sentiment', 'positive_aspects', 'negative_aspects'],
                                'additionalProperties': False
                            },
                            'strict': True
                        }}
)

result = response_json.choices[0].message.content

try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2, ensure_ascii=False)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}\nNot valid JSON")

```json
{
  "sentiment": "mixed",
  "positive_aspects": "πολύ ευχαριστημένος με την κάμερα",
  "negative_aspects": "όχι με την μπαταρία"
}
```

### Λύση 2η. `Response formating` προσπάθεια 2η.

In [10]:
prompt = '''Extract information from this text and return as a valid JSON:

Text: "Ο Γιάννης αγόρασε ένα κινητό και είναι πολύ ευχαριστημένος με την κάμερα αλλά όχι με την μπαταρία."

Do NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON object, nothing else.

Be strict, return only the aspects with ONE WORD, no other words.

JSON:
'''
response_json = client.chat.completions.create(
    model = MODEL,
    messages=[{'role':"user", 'content': prompt}],
    temperature=0.0,
    response_format={"type": "json_schema",
                        'json_schema': {
                            'name': 'sentiment_response',
                            'schema': {
                                'type': 'object',
                                'properties': {
                                    'sentiment': {'type': 'string'},
                                    'positive_aspects': {'type': 'string'},
                                    'negative_aspects': {'type': 'string'}
                                },
                                'required': ['sentiment', 'positive_aspects', 'negative_aspects'],
                                'additionalProperties': False
                            },
                            'strict': True
                        }}
)

result = response_json.choices[0].message.content

try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2, ensure_ascii=False)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}\nNot valid JSON")

```json
{
  "sentiment": "mixed",
  "positive_aspects": "κάμερα",
  "negative_aspects": "μπαταρία"
}
```